<a href="https://colab.research.google.com/github/acadapter123/project2_group7/blob/main/Project2_Group7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets # downloads hugging face sets (yahoo answers)

In [ ]:
import math
import time
import numpy as np
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

bertName = "google-bert/bert-base-uncased"
maxLength = 128
batchSize = 15
learningRate = 2e-5
numEpochs = 5
device = "cuda"
seed = 5
set_seed(seed)
sizeTargetsMB = {"10MB": 10, "100MB": 100, "1GB": 1000}

# loading the data
data = load_dataset("pietrolesci/yahoo_answers_topics")
trainData = data["train"]
testData = data["test"]

print(trainData)
print(testData)
print(trainData[0])



README.md:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

data/train-00000-of-00004-2c55c9d16b60f6(…): reconstructing file:   0%|          |  0.00B /  228MB            

data/train-00000-of-00004-2c55c9d16b60f6(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-ec87b51c64607f(…): reconstructing file:   0%|          |  0.00B /  247MB            

data/train-00001-of-00004-ec87b51c64607f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-f2e3e58859c39f(…): reconstructing file:   0%|          |  0.00B /  267MB            

... (4 outputs left out)

data/test-00000-of-00001-cadb87e1ef8e83f(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1400000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'topic', 'question_title', 'question_content', 'best_answer', 'text', 'uid'],
    num_rows: 1400000
})
Dataset({
    features: ['id', 'topic', 'question_title', 'question_content', 'best_answer', 'text', 'uid'],
    num_rows: 60000
})
{'id': 0, 'topic': 4, 'question_title': "why doesn't an optical mouse work on a glass table?", 'question_content': 'or even on some surfaces?', 'best_answer': 'Optical mice use an LED and a camera to rapidly capture images of the surface beneath the mouse.  The infomation from the camera is analyzed by a DSP (Digital Signal Processor) and used to detect imperfections in the underlying surface and determine motion. Some materials, such as glass, mirrors or other very shiny, uniform surfaces interfere with the ability of the DSP to accurately analyze the surface beneath the mouse.  \\nSince glass is transparent and very uniform, the mouse is unable to pick up enough imperfections in the underlying surface to determine motion. 

In [ ]:
# returns all txt fields in one row combined & byte num to track 10mb, 100mb, 1gb
def addBytes(row):
    return {"numBytes": len(row["text"].encode("utf-8"))}

dropCols = ["id", "question_title", "question_content", "best_answer"]
trainSet = trainData.map(addBytes, num_proc=2, remove_columns=dropCols)
testSet = testData.map(addBytes, num_proc=2, remove_columns=dropCols)

labelCol = "label" if "label" in trainSet.column_names else "topic"
print("Label column:", labelCol)

totalMB = sum(trainSet["numBytes"]) / 1e6
print(f"Full train set: {len(trainSet):,} examples, {totalMB:,.0f} MB of text")

Map (num_proc=2):   0%|          | 0/1400000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/60000 [00:00<?, ? examples/s]

Label column: topic
Full train set: 1,400,000 examples, 729 MB of text


In [ ]:
# building the 10mb, 100mb & 1gb versions
topics = np.array(trainSet[labelCol])
numBytes = np.array(trainSet["numBytes"])
topicIds = np.unique(topics)
rng = np.random.default_rng(seed)
shuffledByTopic = {} # shuffles rows in a topic
for topic in topicIds:
    rowsInTopic = np.where(topics == topic)[0]
    shuffledRows = rng.permutation(rowsInTopic)
    shuffledByTopic[topic] = shuffledRows

def buildVersion(mbSize):
    perTopicBudget = mbSize * 1000000 / len(topicIds)
    chosen = []
    for topic in topicIds:
        idx = shuffledByTopic[topic]
        totBytes = np.cumsum(numBytes[idx])
        cutoff = np.searchsorted(totBytes, perTopicBudget, side="right")
        chosen.append(idx[:max(cutoff, 1)])
    rowNums = np.sort(np.concatenate(chosen))
    return trainSet.select(rowNums)

versions = {} # holds the data for 10mb, 100mb, & 1gb
for name, mb in sizeTargetsMB.items():
    versions[name] = buildVersion(mb)

for name, ds in versions.items():
    print(name, f"{len(ds):,} examples", f"{sum(ds['numBytes']) / 1e6:.1f} MB")

10MB 19,521 examples 10.0 MB
100MB 196,731 examples 100.0 MB
1GB 1,400,000 examples 729.5 MB


In [ ]:
# decides order model sees training rows, easiest to hardest
class CurriculumSampler(torch.utils.data.Sampler):
    def __init__(self, difficulty, startFrac, growthFrac, seed):
        self.order = np.argsort(difficulty, kind="stable")  # row numbers, easiest first
        self.n = len(difficulty)
        self.startFrac = startFrac # how much of the data the model sees as "easy"
        self.growthLen = max(1, int(growthFrac * self.n))
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.n

    def __iter__(self):
        for i in range(self.n):
            progress = min(i / self.growthLen, 1.0)
            frac = self.startFrac + (1 - self.startFrac) * progress   # grows linearly
            poolSize = max(1, int(frac * self.n))
            yield int(self.order[self.rng.integers(0, poolSize)])

class CurriculumTrainer(Trainer): # a Trainer that uses the sampler
    def __init__(self, *args, difficulty, startFrac, growthFrac, sampleSeed, **kwargs):
        super().__init__(*args, **kwargs)
        self.sampler = CurriculumSampler(difficulty, startFrac, growthFrac, sampleSeed)

    def _get_train_sampler(self, *args, **kwargs):
        return self.sampler

In [ ]:
trainSeed = 42
trainBatch = 4
accumSteps = 4
trainEpochs = 1

def runTraining(kind, versionName, startFrac=0.3, growthFrac=0.5):
    set_seed(trainSeed)
    trainVersion = versions[versionName]
    difficulty = np.array(trainVersion["numBytes"])

    tokenizer = AutoTokenizer.from_pretrained(bertName)
    model = AutoModelForMaskedLM.from_pretrained(bertName)

    def tokenizeFn(batch):
        return tokenizer(batch["text"], truncation=True, max_length=maxLength,
                         return_special_tokens_mask=True)

    trainTok = trainVersion.map(tokenizeFn, batched=True, remove_columns=trainVersion.column_names)
    testTok = testSet.map(tokenizeFn, batched=True, remove_columns=testSet.column_names)

    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

    args = TrainingArguments(
        output_dir="out",
        eval_strategy="epoch",
        num_train_epochs=trainEpochs,
        per_device_train_batch_size=trainBatch,
        per_device_eval_batch_size=64,
        gradient_accumulation_steps=accumSteps,
        learning_rate=learningRate,
        weight_decay=0.01,
        logging_steps=10,
        seed=trainSeed,
        data_seed=trainSeed,
        fp16=torch.cuda.is_available(),
        save_strategy="no",
        report_to="none",
    )

    if kind == "baseline":
        trainer = Trainer(model=model, args=args, train_dataset=trainTok,
                          eval_dataset=testTok, processing_class=tokenizer,
                          data_collator=collator)
    else:
        trainer = CurriculumTrainer(model=model, args=args, train_dataset=trainTok,
                                    eval_dataset=testTok, processing_class=tokenizer,
                                    data_collator=collator, difficulty=difficulty,
                                    startFrac=startFrac, growthFrac=growthFrac,
                                    sampleSeed=trainSeed)

    trainer.train()
    evalLoss = trainer.evaluate()["eval_loss"]
    return evalLoss, trainer.state.log_history

In [ ]:
baselineLoss, baselineHistory = runTraining("baseline", "10MB")
curriculumLoss, curriculumHistory = runTraining("curriculum", "10MB")

print("Baseline test loss:  ", round(baselineLoss, 4))
print("Curriculum test loss:", round(curriculumLoss, 4))

startResults = {} # tuning
for startFrac in [0.1, 0.3, 0.5, 1.0]:
    loss, history = runTraining("curriculum", "10MB", startFrac=startFrac)
    startResults[startFrac] = loss
    print(startFrac, round(loss, 4))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

Training Loss,Validation Loss,Epoch
9.342048,2.185082,1


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
